# 10 · Choosing hard data adaptively (RQ3)

*Does adaptively selected hard data beat random selection and hard-example mining at the same data budget?*

The selector is a multi-armed bandit (multiplicative weights; one-step RL with no state). Arms are
condition bins of the 3D-occlusion pool (hidden-body fraction 5–20 %, 20–40 %, 40–60 %, 60–100 %). Each
round it probes every arm with the current model and shifts the data budget toward the bins with the
highest loss. Baselines: uniform random, and hard-example mining (score 4× candidates, keep the hardest).

Same for every strategy: images per round, rounds, optimiser steps, LoRA rank, starting point (the
released HMR2.0 with a fresh adapter). Probing and mining cost forward passes only, never training images.
~1 h on a T4 for 3 strategies × 3 seeds.

Live Blender generation would only change where a bin's images come from (`ArmPool.sample`); with the
existing pool the whole comparison fits in one session.

In [ ]:
STRATEGIES, SEEDS = ["random", "hem", "bandit"], [0, 1, 2]
ROUNDS, ROUND_SIZE, STEPS_PER_ROUND = 8, 250, 60      # 2,000 training images, 480 steps per run
R_LORA, LORA_BLOCKS, LR, BATCH = 16, 4, 1e-4, 16

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/CS747_project"
CODE_DIR    = f"{PROJECT_DIR}/code/rw2b"          # the rw2b folder (pose_lib/, blender/, notebooks/)

from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time, gc, shutil, subprocess
import numpy as np
if not os.path.exists(f"{CODE_DIR}/pose_lib"):              # first run: unpack the code next to the zip
    !cd "{PROJECT_DIR}/code" && unzip -oq rw2b.zip
sys.path.insert(0, CODE_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# HMR2.0 without PHALP/detectron2 (the install that worked in notebook 05 on Python 3.13)
%cd /content
![ -d 4D-Humans ] || git clone -q https://github.com/shubham-goel/4D-Humans.git
%cd /content/4D-Humans
!pip install -q numpy yacs smplx pytorch_lightning timm einops omegaconf trimesh webdataset pyrender
!pip install -q --no-build-isolation git+https://github.com/mattloper/chumpy
!pip install -q --no-deps .
!sed -i 's/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg)/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg, weights_only=False)/' hmr2/models/__init__.py
%cd /content
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, "/content/4D-Humans")

from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import download_models
download_models(CACHE_DIR_4DHUMANS)                           # ~2.5 GB, cached for the session

# SMPL: plain-numpy files only, chumpy never imported (the fix for the broken environment)
from pose_lib.smpl import pkl_to_npz, setup_hmr2_smpl
SMPL_NPZ = f"{PROJECT_DIR}/data/smpl_neutral.npz"
if not os.path.exists(SMPL_NPZ):
    print(pkl_to_npz(f"{PROJECT_DIR}/body_models/smpl/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl", SMPL_NPZ))
setup_hmr2_smpl(SMPL_NPZ)

import torch
from pose_lib import hmr2_ft as ft, lora, metrics, stats
fk = ft.SMPLJoints(SMPL_NPZ)
print("HMR2.0 + SMPL ready")

In [ ]:
# synthetic AMASS dataset from notebook 04 (B_clean / C_paste2d / D_occ3d, train + test)
DATA = "/content/dataset"
if not os.path.exists(f"{DATA}/test/D_occ3d/labels.npz"):
    !tar -xf "{PROJECT_DIR}/data/synth_amass.tar" -C /content
# restore self-occlusion labels if notebook 06 added them in an earlier session
for c in ["B_clean", "C_paste2d", "D_occ3d"]:
    saved = f"{PROJECT_DIR}/data/test_labels_selfocc_{c}.npz"
    if os.path.exists(saved):
        shutil.copy(saved, f"{DATA}/test/{c}/labels.npz")
HAS_SELFOCC = "selfvis_joint" in np.load(f"{DATA}/test/D_occ3d/labels.npz").files
print(os.listdir(f"{DATA}/train"), "| self-occlusion labels:", HAS_SELFOCC)

In [ ]:
def fresh_model():
    m = ft.load_model()
    gc.collect(); torch.cuda.empty_cache()
    return m

In [ ]:
from pose_lib import selector
OUT = f"{PROJECT_DIR}/results/rq3_r{R_LORA}_rounds{ROUNDS}x{ROUND_SIZE}"; os.makedirs(OUT, exist_ok=True)
tr_idx, _ = ft.split_train_val(f"{DATA}/train/D_occ3d", seed=0)
pool_ds = ft.SynthDataset(f"{DATA}/train/D_occ3d", indices=tr_idx, augment=True)
score_ds = ft.SynthDataset(f"{DATA}/train/D_occ3d", indices=tr_idx, augment=False)
arms, names = selector.occfrac_arms(pool_ds.L["occfrac"])
pool = selector.ArmPool([np.intersect1d(a, pool_ds.idx) for a in arms], names)
print({n: len(a) for n, a in zip(pool.names, pool.arms)})
tests = {c: ft.SynthDataset(f"{DATA}/test/{c}") for c in ["D_occ3d", "C_paste2d", "B_clean"]}

In [ ]:
for seed in SEEDS:
    for strat in STRATEGIES:
        tag = f"{strat}_s{seed}"
        if all(os.path.exists(f"{OUT}/{tag}__{t}.npz") for t in tests):
            print("[skip]", tag); continue
        model = fresh_model()
        lora.inject(model, lora.hmr2_targets(LORA_BLOCKS), [tag], r=R_LORA)
        torch.manual_seed(seed)
        trainer = selector.AdapterTrainer(model, tag, pool_ds, fk, lr=LR, batch_size=BATCH,
                                          steps_per_round=STEPS_PER_ROUND, device="cuda", seed=seed, score_ds=score_ds)
        log = selector.run(strat, pool, rounds=ROUNDS, round_size=ROUND_SIZE, train_on=trainer.train_on,
                           loss_of=trainer.loss_of, seed=seed)
        occ = pool_ds.L["occfrac"]
        json.dump({"bandit_probs": log["bandit_probs"], "arm_names": log["arm_names"],
                   "mean_occfrac_per_round": [float(occ[c].mean()) for c in log["chosen"]]},
                  open(f"{OUT}/{tag}_log.json", "w"), indent=1)
        lora.set_active(model, tag)
        for t, ds in tests.items():
            print(f"{tag} on {t}:", end=" "); np.savez(f"{OUT}/{tag}__{t}.npz", **ft.evaluate(model, ds, fk))
        del model, trainer

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
R = {(s, sd, t): dict(np.load(f"{OUT}/{s}_s{sd}__{t}.npz")) for s in STRATEGIES for sd in SEEDS for t in tests}
rows = []
for t in tests:
    for s in STRATEGIES:
        per = [metrics.summarize(R[(s, sd, t)]) for sd in SEEDS]
        keys = [k for k in per[0] if k != "n" and all(k in p for p in per)]
        rows.append({"test": t, "strategy": s, **stats.mean_std_table({k: [p[k] for p in per] for k in keys})})
tab = pd.DataFrame(rows).set_index(["test", "strategy"]); tab.to_csv(f"{OUT}/rq3_summary.csv"); tab

In [ ]:
lines = []
for t in tests:
    for other in ["random", "hem"]:
        r = stats.seed_comparison([R[("bandit", sd, t)]["pa_mpjpe"] for sd in SEEDS],
                                  [R[(other, sd, t)]["pa_mpjpe"] for sd in SEEDS])
        lines.append(f"bandit - {other} on {t}: {r['diff_mean']:+.1f} mm per seed {r['per_seed']} "
                     f"[{r['ci_lo']:+.1f}, {r['ci_hi']:+.1f}] robust={r['robust']}")
print("\n".join(lines)); open(f"{OUT}/rq3_comparisons.txt", "w").write("\n".join(lines))

In [ ]:
# What did the bandit choose? (arm probabilities per round, seed 0)
lg = json.load(open(f"{OUT}/bandit_s0_log.json"))
p = np.array(lg["bandit_probs"])
plt.figure(figsize=(6, 3.5)); plt.stackplot(range(1, len(p) + 1), p.T, labels=lg["arm_names"])
plt.xlabel("round"); plt.ylabel("share of new data"); plt.legend(loc="upper left", fontsize=8); plt.title("Bandit allocation")
plt.tight_layout(); plt.savefig(f"{OUT}/bandit_allocation.png", dpi=150); plt.show()